<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# DKN : Deep Knowledge-Aware Network for News Recommendation
DKN \[1\] is a deep learning model which incorporates information from knowledge graph for better news recommendation. Specifically, DKN uses TransX \[2\] method for knowledge graph representaion learning, then applies a CNN framework, named KCNN, to combine entity embedding with word embedding and generate a final embedding vector for a news article. CTR prediction is made via an attention-based neural scorer. 
<img src="https://raw.githubusercontent.com/recommenders-team/resources/main/kdd2020/images%2FDKN-introduction-pic.JPG" width="600">

## Properties of DKN:
- DKN is a content-based deep model for CTR prediction rather than traditional ID-based collaborative filtering. 
- It makes use of knowledge entities and common sense in news content via joint learning from semantic-level and knnowledge-level representations of news articles.
- DKN uses an attention module to dynamically calculate a user's aggregated historical representaition.



## Data format:
### DKN takes several files as input as follows:
- training / validation / test files: each line in these files represents one instance. Impressionid is used to evaluate performance within an impression session, so it is only used when evaluating, you can set it to 0 for training data. The format is : <br> 
`[label] [userid] [CandidateNews]%[impressionid] `<br> 
e.g., `1 train_U1 N1%0` <br> 
- user history file: each line in this file represents a users' click history. The `history_size` argument of the model is the max number of user's click history we use. We will automatically keep the last his_size number of user click history, if user's click history is more than his_size, and we will automatically padding 0 if user's click history less than his_size. the format is : <br> 
`[Userid] [newsid1,newsid2...]`<br>
e.g., `train_U1 N1,N2` <br> 
- document feature file:
It contains the word and entity features of news. News article is represented by (aligned) title words and title entities. To take a quick example, a news title may be : Trump to deliver State of the Union address next week , then the title words value may be CandidateNews:34,45,334,23,12,987,3456,111,456,432 and the title entitie value may be: entity:45,0,0,0,0,0,0,0,0,0. Only the first value of entity vector is non-zero due to the word Trump. The title value and entity value is hashed from 1 to n(n is the number of distinct words or entities). Each feature length should be fixed at k(doc_size papameter), if the number of words in document is more than k, you should truncate the document to k words, and if the number of words in document is less than k, you should padding 0 to the end. 
the format is like: <br> 
`[Newsid] [w1,w2,w3...wk] [e1,e2,e3...ek]`
- word embedding/entity embedding/ context embedding files: These are npy files of pretrained embeddings. After loading, each file is a [n+1,k] two-dimensional matrix, n is the number of words(or entities) of their hash dictionary, k is dimension of the embedding, note that we keep embedding 0 for zero padding. 
In this experiment, we used GloVe\[4\] vectors to initialize the word embedding. We trained entity embedding using TransE\[2\] on knowledge graph and context embedding is the average of the entity's neighbors in the knowledge graph.<br>

## Global settings and imports

In [ ]:
import os
import time

from recommenders.models.deeprec.models.pytorch.dkn import DKN

## data paths
Usually we will debug and search hyper-parameters on a small dataset.  You can switch between the small dataset and full dataset by changing the value of `tag`.

In [ ]:
tag = 'small' # small or full
data_path = 'data_folder/my/DKN-training-folder'

EPOCHS = 5
HISTORY_SIZE = 20
BATCH_SIZE = 100
RANDOM_SEED = 42  # Set this to None for non-deterministic result

In [ ]:
train_file = os.path.join(data_path, r'train_{0}.txt'.format(tag))
valid_file = os.path.join(data_path, r'valid_{0}.txt'.format(tag))
test_file = os.path.join(data_path, r'test_{0}.txt'.format(tag))
user_history_file = os.path.join(data_path, r'user_history_{0}.txt'.format(tag))
news_feature_file = os.path.join(data_path, r'../paper_feature.txt')
word_embedding_file = os.path.join(data_path, r'word_embedding.npy')
entity_embedding_file = os.path.join(data_path, r'entity_embedding.npy')
context_embedding_file = os.path.join(data_path, r'context_embedding.npy')
infer_embedding_file = os.path.join(data_path, r'infer_embedding.txt')
model_dir = os.path.join(data_path, 'save_models')

## Create the DKN model
The architecture is set on the constructor and the training knobs on `fit`. The metrics are passed to every evaluation.

In [ ]:
metrics = ['auc']
pairwise_metrics = ['group_auc', 'mean_mrr', 'ndcg@2;4;6']

## Train the DKN model
<img src="https://raw.githubusercontent.com/recommenders-team/resources/main/kdd2020/images%2FDKN-main.JPG" width="600">

In [ ]:
model = DKN(
    news_feature_file=news_feature_file,
    user_history_file=user_history_file,
    word_embedding_file=word_embedding_file,
    entity_embedding_file=entity_embedding_file,
    context_embedding_file=context_embedding_file,
    history_size=HISTORY_SIZE,
    filter_sizes=[1, 2, 3],
    num_filters=50,
    attention_layer_size=32,
    layer_sizes=[300],
    enable_BN=False,
    init_method='uniform',
    init_value=0.01,
    seed=RANDOM_SEED,
)

In [ ]:
t01 = time.time()
print(model.run_eval(valid_file, batch_size=BATCH_SIZE, metrics=metrics, pairwise_metrics=pairwise_metrics))
t02 = time.time()
print((t02-t01)/60)

In [ ]:
model = model.fit(
    train_file,
    valid_file,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=0.001,
    embed_l2=0.0,
    layer_l2=0.0,
    max_grad_norm=0.5,
    metrics=metrics,
    pairwise_metrics=pairwise_metrics,
    model_dir=model_dir,
    save_epoch=1,
)

Now we can test the performance on the test set:

In [ ]:
t01 = time.time()
print(model.run_eval(test_file, batch_size=BATCH_SIZE, metrics=metrics, pairwise_metrics=pairwise_metrics))
t02 = time.time()
print((t02-t01)/60)

## Document embedding inference API
After training, you can get document embedding through this document embedding inference API. The input file format is same with document feature file. The output file fomrat is: `[Newsid] [embedding]`

In [ ]:
model = model.run_get_embedding(news_feature_file, infer_embedding_file, batch_size=BATCH_SIZE)

we compre with DKN performance between using knowledge entities or without using knowledge entities (DKN(-)):

| Models | Group-AUC | MRR |NDCG@2 | NDCG@4 |
| :------| :------: | :------: | :------: | :------ |
| DKN | 0.9557 | 0.8993 | 0.8951 | 0.9123 |
| DKN(-) | 0.9506 | 0.8817 | 0.8758 | 0.8982 |
| LightGCN | 0.8608 | 0.5605 | 0.4975 | 0.5792 |

## Reference
\[1\] Wang, Hongwei, et al. "DKN: Deep Knowledge-Aware Network for News Recommendation." Proceedings of the 2018 World Wide Web Conference on World Wide Web. International World Wide Web Conferences Steering Committee, 2018.<br>
